In [ ]:
import pandas as pd
import requests
from datetime import datetime, timedelta

In [ ]:
ACTUAL_URL = "https://data.elexon.co.uk/bmrs/api/v1/datasets/FUELHH/stream"
FORECAST_URL = "https://data.elexon.co.uk/bmrs/api/v1/datasets/WINDFOR/stream"

In [ ]:
# Delivery days whose day-ahead decision (D-1) falls in January 2026:
# 2 Jan -> 1 Feb 2026 (same range as the app's "Jan 2026 decisions" preset).
start_date = "2026-01-02"
end_date = "2026-02-01"

print("Fetching data from BMRS API... Please wait.")

In [ ]:
actual_res = requests.get(ACTUAL_URL, params={"fuelType": "WIND", "settlementDateFrom": start_date, "settlementDateTo": end_date}, timeout=60)
actual_res.raise_for_status()
actual_df = pd.DataFrame([{"time": item["startTime"], "actual": item["generation"]} for item in actual_res.json()])
actual_df['time'] = pd.to_datetime(actual_df['time'], utc=True)
actual_df = actual_df.drop_duplicates(subset=['time']).sort_values('time').reset_index(drop=True)

In [ ]:
def fetch_forecast_rows(start, end, chunk_days=7):
    """Fetch WINDFOR publications in contiguous publish-time chunks.

    Same window as the backend: publish times from (start - 2 days) to
    (end + 2 days). A month of publications is large, so it is split into
    7-day chunks to avoid request time-outs.
    """
    cursor = datetime.strptime(start, "%Y-%m-%d") - timedelta(days=2)
    final_end = datetime.strptime(end, "%Y-%m-%d") + timedelta(days=2, hours=23, minutes=59)
    rows = []
    while cursor <= final_end:
        nxt = min(cursor + timedelta(days=chunk_days), final_end + timedelta(minutes=1))
        res = requests.get(FORECAST_URL, params={
            "publishDateTimeFrom": cursor.strftime("%Y-%m-%dT%H:%M:%SZ"),
            "publishDateTimeTo": (nxt - timedelta(seconds=1)).strftime("%Y-%m-%dT%H:%M:%SZ"),
        }, timeout=60)
        res.raise_for_status()
        rows.extend(res.json())
        cursor = nxt
    return rows

forecast_df = pd.DataFrame([{"time": item["startTime"], "publishTime": item["publishTime"], "forecast": item["generation"]} for item in fetch_forecast_rows(start_date, end_date)])
forecast_df['time'] = pd.to_datetime(forecast_df['time'], utc=True)
forecast_df['publishTime'] = pd.to_datetime(forecast_df['publishTime'], utc=True)
forecast_df = forecast_df.sort_values(['time', 'publishTime', 'forecast']).drop_duplicates(subset=['time', 'publishTime'], keep='last').reset_index(drop=True)

print("Data Fetch Successful! ")
print(f"Actual Data Rows: {len(actual_df)}")
print(f"Forecast Data Rows: {len(forecast_df)}")

In [ ]:
display(actual_df.head(3))

In [ ]:
import numpy as np

In [ ]:
forecast_df['horizon_hours'] = (forecast_df['time'] - forecast_df['publishTime']).dt.total_seconds() / 3600

In [ ]:
valid_forecasts = forecast_df[(forecast_df['horizon_hours'] >= 0) & (forecast_df['horizon_hours'] <= 48)].copy()

In [ ]:
analysis_df = pd.merge(actual_df, valid_forecasts, on='time', how='inner')

In [ ]:
# Sign convention (same as the app/backend): Forecast Error = Forecast - Actual.
# Positive = over-forecast, negative = under-forecast.
analysis_df['error'] = analysis_df['forecast'] - analysis_df['actual']
analysis_df['abs_error'] = analysis_df['error'].abs() 

print("Data Merged and Error Calculated! ")

### Part 1: Error Characteristics of the Forecast Model
Based on the absolute error between the forecast and actual wind generation (Forecast − Actual, January 2026 decisions: delivery 2 Jan – 1 Feb 2026), we observe the following characteristics:
* **Mean Error (TODO: fill from the output of the metrics cell, MW):** the average absolute deviation of the forecast from actual generation.
* **Median Error (TODO: fill from output, MW):** compare with the mean — a median well below the mean indicates occasional large errors (outliers) pulling the mean upward.
* **P99 Error (TODO: fill from output, MW):** 99% of the time the absolute forecast error is below this value, which describes worst-case prediction scenarios.

**Variation with Forecast Horizon:**
_TODO: describe the trend in the scatter plot below (how mean absolute error changes between short-horizon, 0–10 h, and long-horizon, 40–48 h, forecasts) after re-running on the January 2026 data._

In [ ]:
mean_error = analysis_df['abs_error'].mean()
median_error = analysis_df['abs_error'].median()
p99_error = analysis_df['abs_error'].quantile(0.99)

print("\n--- OVERALL ERROR METRICS ---")
print(f"Mean Error: {mean_error:.2f} MW")
print(f"Median Error: {median_error:.2f} MW")
print(f"P99 Error (99% times error is below this): {p99_error:.2f} MW")

display(analysis_df[['time', 'horizon_hours', 'actual', 'forecast', 'abs_error']].head())

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
horizon_error = analysis_df.groupby('horizon_hours')['abs_error'].mean().reset_index()

In [ ]:
plt.figure(figsize=(12, 6))
plt.scatter(horizon_error['horizon_hours'], horizon_error['abs_error'], color='blue', alpha=0.6, label='Mean Error per Horizon')

plt.title('How Error Changes with Forecast Horizon (0 to 48 Hours)', fontsize=14, fontweight='bold')
plt.xlabel('Forecast Horizon (Hours Before Target Time)', fontsize=12)
plt.ylabel('Mean Absolute Error (MW)', fontsize=12)
plt.xlim(0, 48)
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend()
plt.tight_layout()

plt.show()

### Part 2: Recommendation for Reliable Wind Power Capacity
**Question:** How many MW of wind power can we reliably expect to be available to meet the electricity demand?

**Recommendation:** _TODO: state the reliable range from the P5 / P10 values printed by the last code cell (January 2026 data)._

**Reasoning Derived from First Principles:**
1. **The Flaw of Averages:** Average historical wind generation is highly volatile. If we commit to the average as our "reliable" baseline, the grid will face power deficits roughly 50% of the time when wind speeds drop. Therefore, averages cannot be used for base-load reliability.
2. **Percentile-Based Reliability:** To ensure grid stability, we must plan for worst-case scenarios using percentiles:
   * **P5 Generation (TODO: value, MW):** 95% of the time, actual wind generation was at or above this level. This represents a highly conservative, reliable baseload equivalent.
   * **P10 Generation (TODO: value, MW):** 90% of the time, generation was at or above this level, representing a slightly riskier but acceptable capacity commitment.
3. **Conclusion:** Relying on the P5 level ensures ~95% reliability from wind alone. Any demand above this baseline requires backup generation sources (e.g., battery storage, gas peakers) to compensate for wind intermittency.

In [ ]:



actual_generation = actual_df['actual']

mean_gen = actual_generation.mean()
min_gen = actual_generation.min()
max_gen = actual_generation.max()

p5_gen = np.percentile(actual_generation, 5) 

p10_gen = np.percentile(actual_generation, 10)

print("--- ACTUAL WIND GENERATION STATS ---")
print(f"Minimum Generated: {min_gen:.2f} MW")
print(f"Average Generated: {mean_gen:.2f} MW")
print(f"Maximum Generated: {max_gen:.2f} MW")
print("-" * 30)
print(f"P5 Generation (95% Reliability): {p5_gen:.2f} MW")
print(f"P10 Generation (90% Reliability): {p10_gen:.2f} MW")